## TP02 
$\text{Name: PRIM VEASNA}\\
\text{ID: e20221402}
$


## Build the CSVReader Class (OOP Foundation)

### Objective:
- Reinforce OOP fundamentals: classes, constructors, methods, encapsulation.
- Learn to design reusable data classes for DS projects.

Concept:
A CSVReader class should encapsulate all functionality related to reading CSV files.
Instructions:



### Step 1-4: Define the `CSVReader` class

- `__init__(file_path)` stores the path and prepares a private attribute `_data` (encapsulation: the DataFrame is kept inside the object).
- `read()` loads the CSV with `pd.read_csv` and returns the DataFrame.
- `preview(n)` prints the top `n` rows (it reads the file first if it has not been loaded yet).
- `data` is a read-only property so users can access the DataFrame without replacing it directly.

In [1]:
import pandas as pd
import os
class CSVReader:
  

    def __init__(self, file_path):
        self.file_path = file_path
        self._data = None  

    def read(self):
        
        if not os.path.exists(self.file_path):
            raise FileNotFoundError(f"File not found: {self.file_path}")
        self._data = pd.read_csv(self.file_path)
        return self._data

    def preview(self, n=5):
       
        if self._data is None:
            self.read()
        print(f"Top {n} rows of '{self.file_path}':")
        print(self._data.head(n))

    @property
    def data(self):
        
        return self._data

    def __repr__(self):
        status = "loaded" if self._data is not None else "not loaded"
        return f"CSVReader(file_path='{self.file_path}', status={status})"


### Test Simple Data

In [2]:
data = CSVReader("sample_data.csv")
data.preview(5)


Top 5 rows of 'sample_data.csv':
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
3   4    David   NaN        NaN       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0


In [3]:
df = data.read()
print (data)
print("shape: ", df.shape)

CSVReader(file_path='sample_data.csv', status=loaded)
shape:  (7, 7)


In [4]:
print(type(data.data))
print("Missing values per column:")
print(data.data.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
Missing values per column:
id           0
name         0
age          3
height_cm    1
weight_kg    1
city         0
score        2
dtype: int64


# Exercise 2: Strategy Pattern for Missing Values Handling

**Objective:** Learn the Strategy Pattern - one interface, many interchangeable behaviors.

**Concept:** Different strategies to handle missing data (drop, fill mean, fill mode, etc.) that can be flexibly switched without changing the main code.

### Step 1: Import Libraries

In [5]:
from abc import ABC, abstractmethod
from typing import Protocol
import numpy as np

In [6]:
print("Original Dataset:")
print(df)
print("\nMissing Values Count:")
print(df.isnull().sum())
print("\nData Types:")
print(df.dtypes)

Original Dataset:
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
3   4    David   NaN        NaN       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0
5   6    Frank  30.0      175.0       85.0  Los Angeles    NaN
6   7    Grace   NaN      162.0       60.0      Chicago   92.0

Missing Values Count:
id           0
name         0
age          3
height_cm    1
weight_kg    1
city         0
score        2
dtype: int64

Data Types:
id             int64
name          object
age          float64
height_cm    float64
weight_kg    float64
city          object
score        float64
dtype: object


### Step 2 Define the Strategy Pattern

#### Abstract Base Class: MissingValueStrategy

In [7]:
class MissingValueStrategy(ABC):
    """Abstract base class for missing value handling strategies."""
    
    @abstractmethod
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        """Handle missing values in the dataframe.
        
        Args:
            df: Input dataframe with missing values
            
        Returns:
            DataFrame with missing values handled
        """
        pass
    
    def __str__(self):
        return self.__class__.__name__

### Strategy 1: DropMissing

In [8]:

class DropMissing(MissingValueStrategy):
    """Strategy to drop rows with any missing values."""
    
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        """Drop all rows containing missing values."""
        result = df.dropna()
        print(f"Dropped {len(df) - len(result)} rows with missing values.")
        return result

### Strategy 2: FillMean

In [9]:
class FillMean(MissingValueStrategy):
    """Strategy to fill missing values with the mean of numeric columns."""
    
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        """Fill missing values with column means for numeric columns."""
        result = df.copy()
        
        # Get numeric columns only
        numeric_cols = result.select_dtypes(include=[np.number]).columns
        
        for col in numeric_cols:
            if result[col].isnull().any():
                mean_value = result[col].mean()
                result[col].fillna(mean_value, inplace=True)
                print(f"Filled {col} with mean: {mean_value:.2f}")
        
        # For non-numeric columns, we could drop rows or use a different strategy
        # Here we'll just leave them as is
        return result

### Strategy 3: FillMode

In [10]:
class FillMode(MissingValueStrategy):
    """Strategy to fill missing values with the mode (most frequent value)."""
    
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        """Fill missing values with column mode."""
        result = df.copy()
        
        for col in result.columns:
            if result[col].isnull().any():
                mode_value = result[col].mode()
                if len(mode_value) > 0:
                    result[col].fillna(mode_value[0], inplace=True)
                    print(f"Filled {col} with mode: {mode_value[0]}")
        
        return result

### Step 3: DataCleaner Class

This class uses composition to apply the chosen strategy.

In [11]:
class DataCleaner:
    """Context class that applies a missing value handling strategy."""
    
    def __init__(self, strategy: MissingValueStrategy):
        """Initialize with a specific strategy.
        
        Args:
            strategy: A MissingValueStrategy instance
        """
        self._strategy = strategy
    
    def set_strategy(self, strategy: MissingValueStrategy):
        """Change the strategy at runtime.
        
        Args:
            strategy: A new MissingValueStrategy instance
        """
        self._strategy = strategy
        print(f"Strategy changed to: {strategy}")
    
    def clean_data(self, df: pd.DataFrame) -> pd.DataFrame:
        """Clean the dataframe using the current strategy.
        
        Args:
            df: Input dataframe
            
        Returns:
            Cleaned dataframe
        """
        print(f"\nApplying strategy: {self._strategy}")
        print("-" * 50)
        return self._strategy.handle(df)

### Step 4: Test Different Strategies

### Test 1: Drop Missing Strategy


In [12]:
# Create a cleaner with DropMissing strategy
cleaner = DataCleaner(DropMissing())
df_dropped = cleaner.clean_data(df)

print("\nResult after dropping missing values:")
print(df_dropped)
print(f"\nShape: {df_dropped.shape}")


Applying strategy: DropMissing
--------------------------------------------------
Dropped 5 rows with missing values.

Result after dropping missing values:
   id   name   age  height_cm  weight_kg      city  score
0   1  Alice  29.0      165.0       68.0  New York   85.0
4   5    Eva  27.0      160.0       55.0  New York   88.0

Shape: (2, 7)


### Test 2: Fill Mean Strategy

In [13]:
# Change strategy to FillMean
cleaner.set_strategy(FillMean())
df_mean_filled = cleaner.clean_data(df)

print("\nResult after filling with mean:")
print(df_mean_filled)
print("\nMissing values after filling with mean:")
print(df_mean_filled.isnull().sum())

Strategy changed to: FillMean

Applying strategy: FillMean
--------------------------------------------------
Filled age with mean: 30.25
Filled height_cm with mean: 167.00
Filled weight_kg with mean: 70.00
Filled score with mean: 86.00

Result after filling with mean:
   id     name    age  height_cm  weight_kg         city  score
0   1    Alice  29.00      165.0       68.0     New York   85.0
1   2      Bob  30.25      172.0       70.0  Los Angeles   90.0
2   3  Charlie  35.00      168.0       72.0      Chicago   86.0
3   4    David  30.25      167.0       80.0      Houston   75.0
4   5      Eva  27.00      160.0       55.0     New York   88.0
5   6    Frank  30.00      175.0       85.0  Los Angeles   86.0
6   7    Grace  30.25      162.0       60.0      Chicago   92.0

Missing values after filling with mean:
id           0
name         0
age          0
height_cm    0
weight_kg    0
city         0
score        0
dtype: int64


C:\Users\Admin\AppData\Local\Temp\ipykernel_14204\940559035.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  result[col].fillna(mean_value, inplace=True)


### Test 3: Fill Mode Strategy

In [14]:
# Change strategy to FillMode
cleaner.set_strategy(FillMode())
df_mode_filled = cleaner.clean_data(df)

print("\nResult after filling with mode:")
print(df_mode_filled)
print("\nMissing values after filling with mode:")
print(df_mode_filled.isnull().sum())

Strategy changed to: FillMode

Applying strategy: FillMode
--------------------------------------------------
Filled age with mode: 27.0
Filled height_cm with mode: 160.0
Filled weight_kg with mode: 55.0
Filled score with mode: 75.0

Result after filling with mode:
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob  27.0      172.0       55.0  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago   75.0
3   4    David  27.0      160.0       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0
5   6    Frank  30.0      175.0       85.0  Los Angeles   75.0
6   7    Grace  27.0      162.0       60.0      Chicago   92.0

Missing values after filling with mode:
id           0
name         0
age          0
height_cm    0
weight_kg    0
city         0
score        0
dtype: int64


C:\Users\Admin\AppData\Local\Temp\ipykernel_14204\4106858363.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  result[col].fillna(mode_value[0], inplace=True)


### Step 5: Comparison of Strategies

In [15]:
print("=" * 60)
print("COMPARISON OF STRATEGIES")
print("=" * 60)

print(f"\nOriginal Dataset: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Missing values: {df.isnull().sum().sum()}")

print(f"\n1. DropMissing: {df_dropped.shape[0]} rows remaining")
print(f"   Data loss: {((df.shape[0] - df_dropped.shape[0]) / df.shape[0] * 100):.1f}%")

print(f"\n2. FillMean: {df_mean_filled.shape[0]} rows, {df_mean_filled.isnull().sum().sum()} missing values")
print(f"   Numeric columns filled with mean")

print(f"\n3. FillMode: {df_mode_filled.shape[0]} rows, {df_mode_filled.isnull().sum().sum()} missing values")
print(f"   All columns filled with mode (most frequent value)")

COMPARISON OF STRATEGIES

Original Dataset: 7 rows, 7 columns
Missing values: 7

1. DropMissing: 2 rows remaining
   Data loss: 71.4%

2. FillMean: 7 rows, 0 missing values
   Numeric columns filled with mean

3. FillMode: 7 rows, 0 missing values
   All columns filled with mode (most frequent value)


### Bonus: Create a Custom Strategy

In [16]:
class FillMedian(MissingValueStrategy):
    """Strategy to fill missing values with the median of numeric columns."""
    
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        """Fill missing values with column medians for numeric columns."""
        result = df.copy()
        
        numeric_cols = result.select_dtypes(include=[np.number]).columns
        
        for col in numeric_cols:
            if result[col].isnull().any():
                median_value = result[col].median()
                result[col].fillna(median_value, inplace=True)
                print(f"Filled {col} with median: {median_value:.2f}")
        
        return result

# Test the custom strategy
cleaner.set_strategy(FillMedian())
df_median_filled = cleaner.clean_data(df)

print("\nResult after filling with median:")
print(df_median_filled)

Strategy changed to: FillMedian

Applying strategy: FillMedian
--------------------------------------------------
Filled age with median: 29.50
Filled height_cm with median: 166.50
Filled weight_kg with median: 70.00
Filled score with median: 88.00

Result after filling with median:
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob  29.5      172.0       70.0  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago   88.0
3   4    David  29.5      166.5       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0
5   6    Frank  30.0      175.0       85.0  Los Angeles   88.0
6   7    Grace  29.5      162.0       60.0      Chicago   92.0


C:\Users\Admin\AppData\Local\Temp\ipykernel_14204\2426729673.py:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  result[col].fillna(median_value, inplace=True)
C:\Users\Admin\AppData\Local\Temp\ipykernel_14204\2426729673.py:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For exa

### Summary

**Benefits of the Strategy Pattern:**

1. **Flexibility:** Easy to switch between different algorithms at runtime
2. **Extensibility:** New strategies can be added without modifying existing code
3. **Encapsulation:** Each strategy is encapsulated in its own class
4. **Open/Closed Principle:** Open for extension, closed for modification
5. **Single Responsibility:** Each strategy class has one reason to change

**When to use:**
- Multiple algorithms for the same task
- Need to switch algorithms at runtime
- Want to avoid conditional statements for algorithm selection

## Exercise 3

In [17]:
import pandas as pd
import time
from functools import wraps

In [18]:
# 1. Decorator for logging
def log_action(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[START] {func.__name__}")
        result = func(*args, **kwargs)
        print(f"[FINISH] {func.__name__}")
        return result
    return wrapper

In [19]:
# 2. Decorator for execution time
def log_time(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        end = time.perf_counter()

        print(f"[TIME] {func.__name__}: {end - start:.6f} seconds")
        return result
    return wrapper

In [20]:
# 3. CSVReader class from Exercise 1
class CSVReader:
    def __init__(self, file_path):
        self.file_path = file_path
        self.df = None

    @log_action
    @log_time
    def read(self):
        self.df = pd.read_csv(self.file_path)
        return self.df

    @log_action
    def preview(self, n=5):
        if self.df is None:
            self.read()
        print(self.df.head(n))

In [21]:
# 4. Test with sample CSV
reader = CSVReader("sample_data.csv")

df = reader.read()
reader.preview(5)

[START] read
[TIME] read: 0.006504 seconds
[FINISH] read
[START] preview
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
3   4    David   NaN        NaN       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0
[FINISH] preview


## Exercise 4: Implement Factory Pattern for Data Transformations

##### **Objective:**
+ Practice Factory Pattern for scalable creation of transformation objects.
+ Apply abstraction and composition.
##### **Concept:** Instead of manually creating objects, use a factory that decides what transformation to apply.
##### **Instructions:**
1.  Create an abstract class DataTransform with a method apply(df).
2.  Implement subclasses:
    + NormalizeColumns
    + RemoveDuplicates
    + StandardizeText
3.  Create a TransformFactory that returns transformation objects based on string input.

### How Decorator works. 

```python
def my_decorator(func): 

    def mainfunc(*args): 
        print("Before calling the function")
        response = func(*args)
        print("After calling the function")
        return response

    return mainfunc


@my_decorator
def fetch_data(url:str, path: str): 
    return f"Fetching data from {url} and saving to {path}"


response = fetch_data("https://example.com/data", "/data/file.txt")
print(response)
```

### **Decorator** is a function that wrap another function and we can reuser it again.
```output 
    Before calling the function
    After calling the function
    Fetching data from https://example.com/data and saving to /data/file.txt
```

In [22]:
# 1. Abstract Class
class DataTransform(ABC):
    @abstractmethod
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        pass

In [23]:
# 2. Subclasses
class NormalizeColumns(DataTransform):
    """Makes column names lowercase and replaces spaces with underscores."""
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        df.columns = [col.strip().lower().replace(" ", "_") for col in df.columns]
        return df

In [24]:
class RemoveDuplicates(DataTransform):
    """Removes duplicate rows from the DataFrame."""
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.drop_duplicates().reset_index(drop=True)

In [25]:
class StandardizeText(DataTransform):
    """Trims extra spaces and converts text values to lowercase."""
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        text_cols = df.select_dtypes(include=["object", "string"]).columns
        for col in text_cols:
            df[col] = df[col].astype(str).str.strip().str.lower()
        return df

In [26]:
# 3. Factory Class
class TransformFactory:
    @staticmethod
    def create_transform(transform_type: str) -> DataTransform:
        transforms = {
            "normalize_columns": NormalizeColumns,
            "remove_duplicates": RemoveDuplicates,
            "standardize_text": StandardizeText,
        }
        key = transform_type.strip().lower()
        if key not in transforms:
            raise ValueError(f"Unknown transformation: '{transform_type}'")
        return transforms[key]()

In [27]:
pipeline = ["normalize_columns", "standardize_text", "remove_duplicates"]

for step in pipeline:
    transformer = TransformFactory.create_transform(step)
    df = transformer.apply(df)

print(df)

   id     name   age  height_cm  weight_kg         city  score
0   1    alice  29.0      165.0       68.0     new york   85.0
1   2      bob   NaN      172.0        NaN  los angeles   90.0
2   3  charlie  35.0      168.0       72.0      chicago    NaN
3   4    david   NaN        NaN       80.0      houston   75.0
4   5      eva  27.0      160.0       55.0     new york   88.0
5   6    frank  30.0      175.0       85.0  los angeles    NaN
6   7    grace   NaN      162.0       60.0      chicago   92.0


## Exercise 5: Combine and implement the reusable code from previous exercises

In [28]:
from pathlib import Path
from typing import Any, Dict, List, Optional, Union
class DataPipeline(ABC):

    def __init__(self, source: Path, destination: Path) -> None:
        self.source = source
        self.destination = destination
        self.df: pd.DataFrame | None = None

    @abstractmethod
    def load(self) -> None: ...

    @abstractmethod
    def clean(self) -> None: ...

    @abstractmethod
    def transform(self) -> None: ...

    @abstractmethod
    def save(self) -> None: ...

    @log_action
    @log_time
    def run(self) -> None:
        LOGGER.info("PIPELINE START: %s to %s", self.source, self.destination)
        self.load()
        self.clean()
        self.transform()
        self.save()
        LOGGER.info("PIPELINE END: result saved in %s", self.destination)

#### Subclass Core Logics Rule Abstract from DataPipeline

In [29]:
class CSVDataPipeline(DataPipeline):

    def __init__(self, source: Path, destination: Path,
                 cleaner: DataCleaner, transforms: list[DataTransform]) -> None:
        super().__init__(source, destination)
        self.reader = CSVReader(source)
        self.cleaner = cleaner
        self.transforms = list(transforms)

    def load(self) -> None:
        self.df = self.reader.read()
        self.reader.preview(3)

    def clean(self) -> None:
        self.df = self.cleaner.clean(self.df)

    def transform(self) -> None:
        for transform in self.transforms:
            LOGGER.info("Transform: %s", type(transform).__name__)
            self.df = transform.apply(self.df)

    def save(self) -> None:
        self.df.to_csv(self.destination, index=False)

In [30]:
def build_pipeline(strategy: MissingValueStrategy, transform_names: list[str]) -> CSVDataPipeline:
    return CSVDataPipeline(
        source=INPUT_PATH,
        destination=OUTPUT_PATH,
        cleaner=DataCleaner(strategy),
        transforms=[TransformFactory.create(name) for name in transform_names],
    )